# Data Entry

In [ ]:
import pandas as pd  # noqa: F401
import sys
import os

from importlib import reload

# Asegurar que el path apunte a la raíz para encontrar el paquete 'src'
root_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if root_path not in sys.path:
    sys.path.append(root_path)

reload_ = False

In [ ]:
"""
Notebook Cell: Database Reset & Modular Seeding
Description: Drops tables, recreates them.
"""


import src.database  # noqa: E402, F401
import src.database.models as models
from src.database import SessionLocal, engine  # noqa: E402, F401
import src.database.seed_geography  # noqa: E402

def reset_and_seed():
    print("Iniciando reset de base de datos...")
    
    # 1. Limpieza total
    src.database.Base.metadata.drop_all(bind=src.database.engine)
    print("✅ Todas las tablas han sido eliminadas.")
    
    # 2. Reconstrucción del esquema
    src.database.Base.metadata.create_all(bind=src.database.engine)
    print("✅ Estructura de tablas recreada correctamente.")

    # 3. Carga de datos maestros usando el nuevo módulo
    db = src.database.SessionLocal()
    try:
        print("Poblando tablas geográficas...")
        src.database.seed_geography.seed_provincias(db)
    except Exception as e:
        db.rollback()
        print(f"❌ Error durante el seeding: {e}")
    finally:
        db.close()

In [ ]:
import os
import pandas as pd
from importlib import reload

if not reload_:
    print("🗄️ Importando datos de la empresa...")
    print("   ➡️ Importando socios comerciales...")
    import src.imports.init_socios as init_socios
    import src.imports.cql.read as read_files
    print("   ➡️ Importando cliente...")
    import src.imports.cql.clients as clients
    print("   ➡️ Importando creditos...")
    import src.imports.cql.credits as credits
    print("   ➡️ Importando cuotas y cobranzas...")
    import src.imports.cql.quota_and_coll as quotas
    reload_ = True
else:
    reset_and_seed()

    print("🗄️ Importando datos de la empresa...")
    print("   ➡️ Importando socios comerciales...")
    reload(init_socios)
    reload(read_files)
    print("   ➡️ Importando cliente...")
    reload(clients)
    print("   ➡️ Importando creditos...")
    reload(credits)
    print("   ➡️ Importando cuotas y cobranzas...")
    reload(quotas)


In [19]:
from IPython.display import display

import src.reports.balances as reports
reload(reports)
from src.database import engine

df = reports.saldos(con_saldo=False, agrupar=True, originador=True)
display(df.map("$ {:,.2f}".format))

df = reports.saldos(con_saldo=True, agrupar=True, originador=True, vencimientos=True)
display(df.map("$ {:,.2f}".format))

df = reports.saldos()

,Capital,Interés,IVA,Total
Originador,,,,
AMUF,"$ 98,922,619.96","$ 105,216,773.66","$ 22,095,522.47","$ 226,234,916.09"
DECRETO 14/12,$ 0.00,$ 0.00,$ 0.00,$ 0.00
PENALTY,$ 0.00,$ 0.00,$ 0.00,$ 0.00


Capital         Interés             IVA  \
Originador Fecha Vencimiento                                                   
AMUF       2025-05-28             $ 7,126.51     $ 13,237.45      $ 2,779.87   
           2025-06-28             $ 8,256.98     $ 12,303.18      $ 2,583.67   
           2025-07-28             $ 9,566.79     $ 11,220.69      $ 2,356.35   
           2025-08-28            $ 51,632.89     $ 64,748.86     $ 13,597.26   
           2025-09-28            $ 63,124.29     $ 60,689.55     $ 12,744.81   
           2025-10-28           $ 161,014.44     $ 64,696.28     $ 13,586.22   
           2025-11-28           $ 202,068.77    $ 104,266.47     $ 21,895.95   
           2025-12-28           $ 232,477.69     $ 93,952.59     $ 19,730.04   
           2026-01-28           $ 328,780.19    $ 140,277.92     $ 29,458.36   
           2026-02-28           $ 393,111.27    $ 312,519.57     $ 65,629.11   
           2026-03-28           $ 134,515.14    $ 295,763.74     $ 62,110.38   
           2026-04-28           $ 225,392.88    $ 555,231.04    $ 116,598.51   
           2026-05-28           $ 566,223.08    $ 998,292.43    $ 209,641.41   
           2026-06-28         $ 3,773,544.66  $ 8,538,361.38  $ 1,793,055.97   
           2026-07-28         $ 4,254,697.90  $ 9,315,999.39  $ 1,956,359.82   
           2026-08-28         $ 4,587,161.57  $ 8,863,130.36  $ 1,861,257.34   
           2026-09-28         $ 5,079,673.81  $ 8,376,459.71  $ 1,759,056.54   
           2026-10-28         $ 5,607,733.25  $ 7,838,049.12  $ 1,645,990.36   
           2026-11-28         $ 5,683,972.34  $ 7,244,496.71  $ 1,521,344.33   
           2026-12-28         $ 5,694,311.52  $ 6,647,596.78  $ 1,395,995.31   
           2027-01-28         $ 5,425,376.89  $ 6,056,177.48  $ 1,271,797.30   
           2027-02-28         $ 5,061,560.16  $ 5,497,801.22  $ 1,154,538.32   
           2027-03-28         $ 4,857,047.70  $ 4,984,810.49  $ 1,046,810.15   
           2027-04-28         $ 4,618,617.96  $ 4,496,191.17    $ 944,200.12   
           2027-05-28         $ 4,289,899.47  $ 4,032,735.00    $ 846,874.33   
           2027-06-28         $ 4,078,392.05  $ 3,605,387.18    $ 757,131.31   
           2027-07-28         $ 3,817,476.08  $ 3,203,018.37    $ 672,633.86   
           2027-08-28         $ 3,912,425.96  $ 2,829,850.77    $ 594,268.65   
           2027-09-28         $ 3,958,992.62  $ 2,451,312.06    $ 514,775.54   
           2027-10-28         $ 3,331,084.07  $ 2,069,157.12    $ 434,522.98   
           2027-11-28         $ 3,565,383.72  $ 1,748,421.15    $ 367,168.44   
           2027-12-28         $ 3,118,378.20  $ 1,405,643.42    $ 295,185.10   
           2028-01-28         $ 2,447,282.43  $ 1,108,332.74    $ 232,749.88   
           2028-02-28         $ 2,724,863.55    $ 878,926.87    $ 184,574.63   
           2028-03-28         $ 2,376,404.44    $ 623,500.75    $ 130,935.17   
           2028-04-28         $ 2,079,013.81    $ 400,738.97     $ 84,155.18   
           2028-05-28         $ 1,474,654.01    $ 205,854.15     $ 43,229.36   
           2028-06-28           $ 721,380.87     $ 67,621.53     $ 14,200.54   

                                        Total  
Originador Fecha Vencimiento                   
AMUF       2025-05-28             $ 23,143.83  
           2025-06-28             $ 23,143.83  
           2025-07-28             $ 23,143.83  
           2025-08-28            $ 129,979.01  
           2025-09-28            $ 136,558.65  
           2025-10-28            $ 239,296.94  
           2025-11-28            $ 328,231.19  
           2025-12-28            $ 346,160.32  
           2026-01-28            $ 498,516.47  
           2026-02-28            $ 771,259.95  
           2026-03-28            $ 492,389.26  
           2026-04-28            $ 897,222.43  
           2026-05-28          $ 1,774,156.92  
           2026-06-28         $ 14,104,962.01  
           2026-07-28         $ 15,527,057.11  
           2026-08-2